# Phase 5 — Cloud-Cover Stratification: When Does SAR Matter?

The question: **does Sentinel-1 add more value when Sentinel-2 is sparse?**

If true, this gives the paper a clear "when does SAR matter" finding that's
operationally relevant: practitioners with frequent cloud cover should weight
their pipeline toward SAR, while those in cloud-free regions can rely on optical.

## Approach

For each (field, harvest_year) record:

1. **Compute its S2 availability** during the relevant growing window (day 200–365,
   roughly Mar–Aug of harvest year). Higher = more cloud-free.
2. **Bin into terciles** (low / medium / high cloud) per LOYO fold.
3. **Compute MAE per (stream, phase, stratum)** using existing Phase 3 LSTM predictions.
4. **Compute S1's marginal value**: `MAE(S2_only) − MAE(S1+S2)`. If S1 helps, this is positive.
   If it grows with cloudiness, S1 adds more value under clouds.

This is purely post-hoc on existing model outputs — no new training.

## Key caveat

This is not a controlled experiment. All fields go through the same model; we're
slicing by per-field cloudiness to see if the model's S1-vs-S2 trade-off shifts.
A more rigorous version would train separate models per stratum, but that adds
50 more fits and the post-hoc finding is usually sufficient for a paper section.


## 1. Setup

In [1]:
from pathlib import Path
import sys, os
PROJECT_ROOT = Path('/fs1/home/atikul/Phenology')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

TENSORS_DIR = PROJECT_ROOT / 'cache' / 'tensors'
LSTM_PREDS  = PROJECT_ROOT / 'phase3_outputs/lstm/lstm_predictions.npz'
OUTPUT_DIR  = PROJECT_ROOT / 'phase5_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

YEARS = [2017, 2018, 2019, 2020, 2021]
STREAMS = ['S1_only', 'S2_only', 'Met_only', 'S1+S2', 'All']
PHASES = [10, 12, 15, 18, 19, 21, 24]

# Window for cloudiness measurement: spring-summer of harvest year
# T=426 starts Sep 1 of (Y-1). Day 200 ≈ Mar 19 of Y; day 365 ≈ Aug 31 of Y.
S2_WINDOW = (200, 365)


## 2. Compute per-field S2 availability

In [2]:
print('Computing per-field S2 observation rates...')

per_field = []
for y in YEARS:
    z = np.load(TENSORS_DIR / f'phenology_{y}.npz', allow_pickle=False)
    feat_names = [str(s) for s in z['feature_names']]
    s2_idx = [feat_names.index(f) for f in ('blue','green','red','nir','swir1','swir2',
                                              're1','re2','re3','EVI','MTCI','NDRE','CIrededge')]
    mask = z['mask']                # (N, T, F)
    xid = z['xid']
    bagid = z['bag_id']
    rank = z['rank']
    yr   = z['harvest_year']

    # S2 observation rate within the spring-summer window
    s2_mask = mask[:, S2_WINDOW[0]:S2_WINDOW[1], s2_idx[0]]   # all S2 bands share same mask
    obs_rate = s2_mask.mean(axis=1)                          # (N,)

    df = pd.DataFrame({
        'xid':   xid,
        'bag_id': bagid,
        'rank':  rank,
        'year':  yr,
        's2_obs_rate': obs_rate.astype(np.float32),
    })
    per_field.append(df)

per_field = pd.concat(per_field, ignore_index=True)
print(f'Per-field records: {len(per_field):,}')
print(f'S2 obs-rate distribution: min={per_field["s2_obs_rate"].min():.3f}, '
      f'median={per_field["s2_obs_rate"].median():.3f}, '
      f'max={per_field["s2_obs_rate"].max():.3f}')

# Distribution by year
print('\n=== S2 obs-rate quartiles per year ===')
print(per_field.groupby('year')['s2_obs_rate']
      .describe(percentiles=[0.25,0.5,0.75])[['min','25%','50%','75%','max']].round(3))


Computing per-field S2 observation rates...
Per-field records: 26,815
S2 obs-rate distribution: min=0.024, median=0.170, max=0.303

=== S2 obs-rate quartiles per year ===
        min    25%    50%    75%    max
year                                   
2017  0.024  0.085  0.115  0.133  0.200
2018  0.067  0.139  0.212  0.230  0.297
2019  0.067  0.133  0.200  0.218  0.285
2020  0.073  0.145  0.212  0.236  0.303
2021  0.042  0.103  0.152  0.170  0.261


## 3. Assign cloudiness terciles (per year × policy to keep balanced strata)

In [3]:
# We compute terciles GLOBALLY across all fields/years for a single consistent set of cut-points.
# This makes the strata comparable across years and means the paper can quote
# a single set of thresholds.

lo_thr = per_field['s2_obs_rate'].quantile(1/3)
hi_thr = per_field['s2_obs_rate'].quantile(2/3)

def assign_stratum(rate):
    if rate < lo_thr: return 'low'    # MOST cloudy
    if rate < hi_thr: return 'med'
    return 'high'                     # LEAST cloudy

per_field['stratum'] = per_field['s2_obs_rate'].apply(assign_stratum)
print(f'Cut-points: low if rate < {lo_thr:.3f}; high if rate ≥ {hi_thr:.3f}')
print('\nStratum sizes:')
print(per_field['stratum'].value_counts().to_string())

# Save for reproducibility
per_field.to_csv(OUTPUT_DIR / 'per_field_s2_strata.csv', index=False)


Cut-points: low if rate < 0.127; high if rate ≥ 0.206

Stratum sizes:
stratum
high    9328
med     9183
low     8304


## 4. Load LSTM predictions and join with strata

In [7]:
print('Loading LSTM predictions...')
z = np.load(LSTM_PREDS, allow_pickle=False)
keys = z.files
print(f'  total arrays: {len(keys)}')

# Show a sample to confirm format
print('Sample keys:')
for k in sorted(keys)[:6]:
    print(f'  {k}: {z[k].shape}')

# Auto-detect the separator used by save_pred_npz
# Expected format: "{stream}__{policy}__{year}__{name}"
def load_fold(stream, policy, year):
    pref = f'{stream}__{policy}__{year}'
    try:
        return {
            'yt':   z[f'{pref}__yt'],
            'yp':   z[f'{pref}__yp'],
            'xid':  z[f'{pref}__xid'],
            'bag':  z[f'{pref}__bag'],
            'rank': z[f'{pref}__rank'],
            'year': z[f'{pref}__year'],
        }
    except KeyError as e:
        # Show available keys with matching prefix to help diagnose
        matching = [k for k in z.files if pref in k or f'{stream}_{policy}_{year}' in k]
        print(f'Could not find expected keys for {pref}. Matching keys:')
        for m in matching[:10]:
            print(f'  {m}')
        raise

sample = load_fold('All', 'all10', 2020)
print(f'\nSample fold (All/all10/2020): n_records={len(sample["xid"])}, yp shape={sample["yp"].shape}')

Loading LSTM predictions...
  total arrays: 300
Sample keys:
  All__all10__2017__bag: (5608,)
  All__all10__2017__rank: (5608,)
  All__all10__2017__xid: (5608,)
  All__all10__2017__year: (5608,)
  All__all10__2017__yp: (5608, 7)
  All__all10__2017__yt: (5608, 7)

Sample fold (All/all10/2020): n_records=5181, yp shape=(5181, 7)


In [6]:
# Diagnostic: print a few real key names
z = np.load(LSTM_PREDS, allow_pickle=False)
print(f'Total arrays: {len(z.files)}')
print('First 12 keys:')
for k in sorted(z.files)[:12]:
    print(f'  {k}: {z[k].shape}  {z[k].dtype}')

Total arrays: 300
First 12 keys:
  All__all10__2017__bag: (5608,)  int64
  All__all10__2017__rank: (5608,)  int32
  All__all10__2017__xid: (5608,)  int64
  All__all10__2017__year: (5608,)  int32
  All__all10__2017__yp: (5608, 7)  int32
  All__all10__2017__yt: (5608, 7)  int32
  All__all10__2018__bag: (5392,)  int64
  All__all10__2018__rank: (5392,)  int32
  All__all10__2018__xid: (5392,)  int64
  All__all10__2018__year: (5392,)  int32
  All__all10__2018__yp: (5392, 7)  int32
  All__all10__2018__yt: (5392, 7)  int32


## 5. Compute per-(field, stream, phase) errors and join with strata

In [8]:
# We use the all10 policy as the basis for this analysis since it's the canonical
# Phase 3 LSTM (and matches the practitioner-deployed model)

POLICY = 'all10'

rows = []
for stream in ['S1_only', 'S2_only', 'S1+S2', 'All']:  # skip Met_only for this analysis
    for y in YEARS:
        try:
            fold = load_fold(stream, POLICY, y)
        except KeyError:
            print(f'  missing fold: {stream}/{POLICY}/{y}, skipping')
            continue
        yt = fold['yt']     # (N, P)
        yp = fold['yp']     # (N, P)
        xid = fold['xid']
        for j, phase in enumerate(PHASES):
            valid = (yt[:, j] >= 0) & (yp[:, j] >= 0)
            for i in np.where(valid)[0]:
                rows.append({
                    'stream':   stream,
                    'phase':    int(phase),
                    'year':     int(y),
                    'xid':      int(xid[i]),
                    'ae':       float(abs(yt[i, j] - yp[i, j])),
                })

errors = pd.DataFrame(rows)
print(f'Per-(field, stream, phase) errors: {len(errors):,}')

# Merge with stratum information
errors = errors.merge(per_field[['xid', 'year', 's2_obs_rate', 'stratum']],
                      on=['xid', 'year'], how='left')
n_missing = errors['stratum'].isna().sum()
if n_missing > 0:
    print(f'WARN: {n_missing} records could not be matched to a stratum (dropping)')
    errors = errors.dropna(subset=['stratum'])
print(f'After merge: {len(errors):,}')


Per-(field, stream, phase) errors: 659,640
After merge: 659,640


## 6. Mean absolute error by (stream, phase, stratum)

In [9]:
# Aggregate
agg = (errors.groupby(['stream','phase','stratum'])
              .agg(mae=('ae','mean'), n=('ae','size'))
              .reset_index())

# Pivot for readable display: rows = (phase, stratum), cols = stream
mae_table = agg.pivot_table(index=['phase','stratum'], columns='stream',
                              values='mae').round(2)
print('=== MAE by (phase × cloudiness stratum × stream) ===')
print('"low" = LOW S2 observation rate (MOST cloudy)')
print('"high" = HIGH S2 observation rate (LEAST cloudy)')
print()
print(mae_table.to_string())


=== MAE by (phase × cloudiness stratum × stream) ===
"low" = LOW S2 observation rate (MOST cloudy)
"high" = HIGH S2 observation rate (LEAST cloudy)

stream           All  S1+S2  S1_only  S2_only
phase stratum                                
10    high     11.01  10.15    10.08     9.99
      low       8.88   8.91    11.44     8.52
      med       9.21   9.16    11.01     8.84
12    high     12.24  10.90    10.99    10.59
      low      10.20   9.81    11.49     9.82
      med      10.52   9.67    10.73     9.68
15    high      8.86   9.09     9.08     8.68
      low       9.82  10.06    10.17    10.85
      med       9.43   9.73    10.06     9.75
18    high      5.39   5.41     6.25     5.71
      low       5.07   5.15     5.57     6.16
      med       5.45   5.81     5.97     5.78
19    high      7.81   7.86     8.33     8.11
      low       7.26   7.25     7.56     7.89
      med       7.74   7.76     8.24     7.98
21    high      7.13   6.98     7.64     7.28
      low       6.93   

## 7. S1's marginal value: MAE(S2_only) − MAE(S1+S2)

In [10]:
# Positive Δ means S1+S2 is better than S2_only — i.e., S1 added value.
# We want to see Δ GROW as cloudiness increases (stratum=low).

s1_marginal = (mae_table['S2_only'] - mae_table['S1+S2']).round(2)
s1_marginal_df = s1_marginal.unstack('stratum')
# Reorder columns: low (cloudy) → med → high (clear)
s1_marginal_df = s1_marginal_df[['low','med','high']]
s1_marginal_df['Δ (cloudy − clear)'] = (s1_marginal_df['low'] - s1_marginal_df['high']).round(2)

print('=== S1\'s marginal value by stratum (positive = S1 helps) ===')
print('Comparing MAE(S2_only) − MAE(S1+S2) in each cloudiness bin')
print()
print(s1_marginal_df.to_string())
print()
print('Δ (cloudy − clear) > 0 means S1 adds MORE value when S2 is sparse.')
print('Δ < 0 means S1 adds more value when S2 is already abundant.')


=== S1's marginal value by stratum (positive = S1 helps) ===
Comparing MAE(S2_only) − MAE(S1+S2) in each cloudiness bin

stratum   low   med  high  Δ (cloudy − clear)
phase                                        
10      -0.39 -0.32 -0.16               -0.23
12       0.01  0.01 -0.31                0.32
15       0.79  0.02 -0.41                1.20
18       1.01 -0.03  0.30                0.71
19       0.64  0.22  0.25                0.39
21       0.47  0.11  0.30                0.17
24       0.38  0.59 -0.06                0.44

Δ (cloudy − clear) > 0 means S1 adds MORE value when S2 is sparse.
Δ < 0 means S1 adds more value when S2 is already abundant.


In [11]:
# Same for: All vs S2_only (full-stack value-add)
all_marginal = (mae_table['S2_only'] - mae_table['All']).round(2)
all_marginal_df = all_marginal.unstack('stratum')[['low','med','high']]
all_marginal_df['Δ (cloudy − clear)'] = (all_marginal_df['low'] - all_marginal_df['high']).round(2)
print('=== Full-stack (All vs S2_only) marginal value by stratum ===')
print(all_marginal_df.to_string())


=== Full-stack (All vs S2_only) marginal value by stratum ===
stratum   low   med  high  Δ (cloudy − clear)
phase                                        
10      -0.36 -0.37 -1.02                0.66
12      -0.38 -0.84 -1.65                1.27
15       1.03  0.32 -0.18                1.21
18       1.09  0.33  0.32                0.77
19       0.63  0.24  0.30                0.33
21      -0.08 -0.07  0.15               -0.23
24       0.23  0.21 -0.29                0.52


In [13]:
"""
Add-on cell for notebook 08: verify the cloud-cover finding is temporally robust.

Paste this as a new cell in notebook 08 after Section 7.

If S1's "Δ (cloudy − clear)" pattern holds for phases 15+ in MOST years
individually, the finding is robust. If it only appears in a couple of years,
we should reframe more carefully.
"""

import numpy as np
import pandas as pd

# Re-aggregate errors but split by year as well
# (assumes `errors` DataFrame from earlier Section 5 is still in scope)
per_year_agg = (errors.groupby(['stream','phase','stratum','year'])
                      .agg(mae=('ae','mean'), n=('ae','size'))
                      .reset_index())

# S1's marginal value per year
rows = []
for y in [2017, 2018, 2019, 2020, 2021]:
    sub = per_year_agg[per_year_agg['year'] == y]
    pivot = sub.pivot_table(index=['phase','stratum'], columns='stream', values='mae')
    if 'S2_only' not in pivot.columns or 'S1+S2' not in pivot.columns:
        continue
    marginal = (pivot['S2_only'] - pivot['S1+S2']).unstack('stratum')
    if not all(c in marginal.columns for c in ['low','med','high']):
        continue
    marginal = marginal[['low','med','high']]
    marginal['Δ_cloudy_clear'] = marginal['low'] - marginal['high']
    marginal['year'] = y
    rows.append(marginal.reset_index())

per_year = pd.concat(rows, ignore_index=True)

print('=== S1 marginal value (cloudy − clear) per phase per year ===')
print('Positive = S1 helps more under clouds (the hypothesis)')
print()
pivot = per_year.pivot(index='phase', columns='year', values='Δ_cloudy_clear').round(2)
pivot['mean'] = pivot.mean(axis=1).round(2)
print(pivot.to_string())

print()
print('=== Fraction of years where Δ_cloudy_clear > 0 per phase ===')
for phase in [10, 12, 15, 18, 19, 21, 24]:
    sub = per_year[per_year['phase'] == phase]
    if len(sub) == 0:
        continue
    n_pos = (sub['Δ_cloudy_clear'] > 0).sum()
    n_tot = len(sub)
    bar = '█' * int(30 * n_pos / max(n_tot, 1))
    print(f'  phase {phase}: {n_pos}/{n_tot}  {bar}')

# Also: are the absolute S1 marginals positive in low-stratum across years?
print()
print('=== S1 marginal value in LOW stratum (cloudy) per phase per year ===')
print('Positive = S1 helps in cloudy conditions')
pivot_low = per_year.pivot(index='phase', columns='year', values='low').round(2)
pivot_low['mean'] = pivot_low.mean(axis=1).round(2)
print(pivot_low.to_string())

=== S1 marginal value (cloudy − clear) per phase per year ===
Positive = S1 helps more under clouds (the hypothesis)

year   2018  2019  2020  2021  mean
phase                              
10    -0.25 -0.17 -0.45  1.05  0.04
12    -0.17  0.05 -0.45  0.40 -0.04
15     1.17 -0.12  0.33 -0.06  0.33
18     0.41  0.22  0.65  0.74  0.50
19    -0.47 -0.25  0.91 -0.43 -0.06
21    -0.14  0.43  0.54  0.64  0.37
24     0.24 -0.01  0.71 -0.38  0.14

=== Fraction of years where Δ_cloudy_clear > 0 per phase ===
  phase 10: 1/4  ███████
  phase 12: 2/4  ███████████████
  phase 15: 2/4  ███████████████
  phase 18: 4/4  ██████████████████████████████
  phase 19: 1/4  ███████
  phase 21: 3/4  ██████████████████████
  phase 24: 2/4  ███████████████

=== S1 marginal value in LOW stratum (cloudy) per phase per year ===
Positive = S1 helps in cloudy conditions
year   2018  2019  2020  2021  mean
phase                              
10    -0.49  0.38 -1.04  0.29 -0.21
12    -1.31 -0.04 -0.08  0.19 -0.31
15  

In [15]:
"""
Add-on cells for notebook 08.

Cell A: Diagnose why 2017 is missing from the per-year breakdown.
Cell B: Try alternate strata definitions to see if the cloud-cover pattern
        becomes more robust.

Paste these as new cells at the END of notebook 08 (after the per-year add-on).
The DataFrames `errors`, `per_field`, `PHASES`, `STREAMS`, `LSTM_PREDS` are
already in scope.
"""

# ============================================================================
# CELL A — Why is 2017 missing?
# ============================================================================
import numpy as np
import pandas as pd

print('=== Where did 2017 go? ===')
print('Years in per_field:', sorted(per_field['year'].unique()))
print('Years in errors   :', sorted(errors['year'].unique()))
print()

# Check that 2017 predictions exist in the LSTM npz
z = np.load(LSTM_PREDS, allow_pickle=False)
for stream in ['S1_only', 'S2_only', 'S1+S2', 'All']:
    yt_key  = f'{stream}__all10__2017__yt'
    xid_key = f'{stream}__all10__2017__xid'
    has_yt  = yt_key  in z.files
    has_xid = xid_key in z.files
    print(f'  {stream}/all10/2017: yt={has_yt}, xid={has_xid}')

# How many predictions for 2017 in `errors`?
n_2017 = (errors['year'] == 2017).sum()
print(f'\nRows in errors with year==2017: {n_2017}')

# If 2017 is in per_field but not errors, it dropped during merge.
# Check the merge join key.
n_per_field_2017 = (per_field['year'] == 2017).sum()
print(f'Rows in per_field with year==2017: {n_per_field_2017}')

# Check: when we merge errors with per_field on (xid, year), how many 2017 errors got matched?
# (errors is already merged in Section 5 of the notebook)
if n_2017 == 0 and n_per_field_2017 > 0:
    print('\n→ 2017 predictions exist in npz but did not survive the merge.')
    print('  This is likely a type mismatch between errors.xid and per_field.xid for 2017.')
    # Specific check: are XID dtypes consistent?
    sample_2017_per_field_xids = set(per_field[per_field['year']==2017]['xid'].head(20).astype(int))
    # Look at raw npz xid for one stream
    try:
        npz_xids = z['S2_only__all10__2017__xid'][:20].astype(int)
        print(f'  First 5 XIDs in per_field (2017): {sorted(sample_2017_per_field_xids)[:5]}')
        print(f'  First 5 XIDs in npz       (2017): {sorted(npz_xids.tolist())[:5]}')
    except KeyError:
        pass

=== Where did 2017 go? ===
Years in per_field: [np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021)]
Years in errors   : [np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]

  S1_only/all10/2017: yt=True, xid=True
  S2_only/all10/2017: yt=True, xid=True
  S1+S2/all10/2017: yt=True, xid=True
  All/all10/2017: yt=True, xid=True

Rows in errors with year==2017: 138432
Rows in per_field with year==2017: 5608


In [16]:
"""
CELL B — Try alternate strata definitions to see if the cloud-cover pattern
becomes more robust.

Three alternate stratifications:
  1. Per-year terciles (controls for inter-annual S2 availability shifts)
  2. Binary cut (top vs bottom half) — fewer strata, more samples per cell
  3. Quintiles (5 bins) — finer granularity to see if effect is at the extremes
"""
import numpy as np
import pandas as pd


# Re-do the merge with all 5 years (use errors_full to include 2017 once Cell A
# tells us what was wrong; for now assume errors has whatever it has)

print(f'Re-running stratification analyses on errors with {len(errors):,} rows '
      f'across years {sorted(errors["year"].unique())}')

# ============================================================================
# STRATIFICATION 1: Per-year terciles
# ============================================================================
print('\n' + '='*72)
print('STRAT 1: Per-year terciles (each year stratified by its own distribution)')
print('='*72)

# Re-assign stratum based on per-year terciles
per_field_pyt = per_field.copy()
def per_year_stratum(group):
    lo = group.quantile(1/3)
    hi = group.quantile(2/3)
    return pd.cut(group, bins=[-np.inf, lo, hi, np.inf], labels=['low','med','high'])
per_field_pyt['stratum_pyt'] = (per_field_pyt.groupby('year')['s2_obs_rate']
                                              .transform(per_year_stratum))
print('Stratum counts by year:')
print(per_field_pyt.groupby(['year','stratum_pyt']).size().unstack().to_string())

# Merge with errors and compute S1 marginal
errors_pyt = errors.merge(per_field_pyt[['xid','year','stratum_pyt']],
                            on=['xid','year'], how='left')
agg_pyt = (errors_pyt.groupby(['stream','phase','stratum_pyt'])['ae']
                      .mean().reset_index()
                      .rename(columns={'ae':'mae'}))
pivot_pyt = agg_pyt.pivot_table(index=['phase','stratum_pyt'], columns='stream', values='mae')

if 'S2_only' in pivot_pyt.columns and 'S1+S2' in pivot_pyt.columns:
    s1_marg_pyt = (pivot_pyt['S2_only'] - pivot_pyt['S1+S2']).unstack('stratum_pyt')
    s1_marg_pyt = s1_marg_pyt[['low','med','high']]
    s1_marg_pyt['Δ (cloudy − clear)'] = (s1_marg_pyt['low'] - s1_marg_pyt['high']).round(2)
    print('\nS1 marginal value (per-year terciles):')
    print(s1_marg_pyt.round(2).to_string())

# Per-year robustness
print('\nPer-year robustness check (per-year terciles):')
rows = []
for y in sorted(errors_pyt['year'].dropna().unique()):
    sub = errors_pyt[errors_pyt['year'] == y]
    pivot_y = (sub.groupby(['stream','phase','stratum_pyt'])['ae'].mean()
                  .reset_index().pivot_table(index=['phase','stratum_pyt'],
                                              columns='stream', values='ae'))
    if 'S2_only' not in pivot_y.columns or 'S1+S2' not in pivot_y.columns:
        continue
    marg_y = (pivot_y['S2_only'] - pivot_y['S1+S2']).unstack('stratum_pyt')
    if not all(c in marg_y.columns for c in ['low','med','high']):
        continue
    marg_y['Δ_cloudy_clear'] = marg_y['low'] - marg_y['high']
    marg_y['year'] = int(y)
    rows.append(marg_y.reset_index())
if rows:
    per_year_marg = pd.concat(rows, ignore_index=True)
    pivot_pyy = per_year_marg.pivot(index='phase', columns='year', values='Δ_cloudy_clear').round(2)
    pivot_pyy['mean'] = pivot_pyy.mean(axis=1).round(2)
    print(pivot_pyy.to_string())

    # Fraction positive
    print('\nFraction of years where Δ_cloudy_clear > 0 (per-year terciles):')
    for phase in sorted(per_year_marg['phase'].unique()):
        sub = per_year_marg[per_year_marg['phase'] == phase]
        n_pos = (sub['Δ_cloudy_clear'] > 0).sum()
        n_tot = len(sub)
        bar = '█' * int(30 * n_pos / max(n_tot, 1))
        print(f'  phase {int(phase)}: {n_pos}/{n_tot}  {bar}')


# ============================================================================
# STRATIFICATION 2: Binary cut (top vs bottom HALF — global)
# ============================================================================
print('\n' + '='*72)
print('STRAT 2: Binary cut (top 50% vs bottom 50% global)')
print('='*72)

median = per_field['s2_obs_rate'].median()
per_field_bin = per_field.copy()
per_field_bin['stratum_bin'] = np.where(per_field_bin['s2_obs_rate'] < median, 'low', 'high')
print(f'Median cut: {median:.3f}')
print(f'Bin sizes: {per_field_bin["stratum_bin"].value_counts().to_string()}')

errors_bin = errors.merge(per_field_bin[['xid','year','stratum_bin']],
                            on=['xid','year'], how='left')
agg_bin = (errors_bin.groupby(['stream','phase','stratum_bin'])['ae']
                      .mean().reset_index()
                      .rename(columns={'ae':'mae'}))
pivot_bin = agg_bin.pivot_table(index=['phase','stratum_bin'], columns='stream', values='mae')

if 'S2_only' in pivot_bin.columns and 'S1+S2' in pivot_bin.columns:
    s1_marg_bin = (pivot_bin['S2_only'] - pivot_bin['S1+S2']).unstack('stratum_bin')
    s1_marg_bin['Δ (low − high)'] = (s1_marg_bin['low'] - s1_marg_bin['high']).round(2)
    print('\nS1 marginal value (binary, low=cloudy half):')
    print(s1_marg_bin.round(2).to_string())

# Per-year robustness for binary
print('\nPer-year robustness (binary):')
rows = []
for y in sorted(errors_bin['year'].dropna().unique()):
    sub = errors_bin[errors_bin['year'] == y]
    pivot_y = (sub.groupby(['stream','phase','stratum_bin'])['ae'].mean()
                  .reset_index().pivot_table(index=['phase','stratum_bin'],
                                              columns='stream', values='ae'))
    if 'S2_only' not in pivot_y.columns or 'S1+S2' not in pivot_y.columns:
        continue
    marg_y = (pivot_y['S2_only'] - pivot_y['S1+S2']).unstack('stratum_bin')
    if not all(c in marg_y.columns for c in ['low','high']):
        continue
    marg_y['Δ_cloudy_clear'] = marg_y['low'] - marg_y['high']
    marg_y['year'] = int(y)
    rows.append(marg_y.reset_index())
if rows:
    per_year_marg_b = pd.concat(rows, ignore_index=True)
    pivot_b = per_year_marg_b.pivot(index='phase', columns='year', values='Δ_cloudy_clear').round(2)
    pivot_b['mean'] = pivot_b.mean(axis=1).round(2)
    print(pivot_b.to_string())

    print('\nFraction of years where Δ_cloudy_clear > 0 (binary):')
    for phase in sorted(per_year_marg_b['phase'].unique()):
        sub = per_year_marg_b[per_year_marg_b['phase'] == phase]
        n_pos = (sub['Δ_cloudy_clear'] > 0).sum()
        n_tot = len(sub)
        bar = '█' * int(30 * n_pos / max(n_tot, 1))
        print(f'  phase {int(phase)}: {n_pos}/{n_tot}  {bar}')

print('\n' + '='*72)
print('Interpretation:')
print('='*72)
print("""
Look at the 'Fraction of years > 0' tables for each stratification:

- If per-year terciles gives MORE consistent results than global terciles
  → inter-annual S2 variation was hiding the effect; the per-year stratification
  is the correct framing for the paper.

- If binary cut gives more robust results
  → the effect is at the extremes (very cloudy vs very clear) and finer
  stratification just added noise.

- If neither helps
  → the cloud-cover dependence is truly weak; commit to Option A (narrow claim
  to phase 18, possibly 21) or Option B (drop the cloud-cover claim).
""")

Re-running stratification analyses on errors with 659,640 rows across years [np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]

STRAT 1: Per-year terciles (each year stratified by its own distribution)
Stratum counts by year:
stratum_pyt   low   med  high
year                         
2017         2018  1826  1764
2018         1807  1821  1764
2019         2106  1782  1653
2020         1830  1946  1405
2021         1836  1679  1578


/tmp/atikul-tmpdir-1QD7ds/ipykernel_952441/4207194529.py:36: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(per_field_pyt.groupby(['year','stratum_pyt']).size().unstack().to_string())
/tmp/atikul-tmpdir-1QD7ds/ipykernel_952441/4207194529.py:41: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  agg_pyt = (errors_pyt.groupby(['stream','phase','stratum_pyt'])['ae']
/tmp/atikul-tmpdir-1QD7ds/ipykernel_952441/4207194529.py:44: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain th


S1 marginal value (per-year terciles):
stratum_pyt   low   med  high  Δ (cloudy − clear)
phase                                            
10          -0.37 -0.39 -0.08               -0.29
12          -0.06 -0.24  0.01               -0.07
15           0.35  0.04 -0.10                0.45
18           0.70  0.32  0.16                0.54
19           0.47  0.33  0.26                0.22
21           0.43  0.29  0.09                0.35
24           0.41  0.19  0.29                0.12

Per-year robustness check (per-year terciles):
year   2017  2018  2019  2020  2021  mean
phase                                    
10     0.61 -0.99 -0.65 -0.72  0.18 -0.31
12     0.12 -0.54 -0.04 -0.47  0.33 -0.12
15     1.38  0.68 -0.15  0.22 -0.10  0.41
18     1.85  0.30  0.09  0.13  0.32  0.54
19     1.17 -0.21 -0.35  0.61 -0.17  0.21
21     0.73  0.17  0.37  0.37  0.02  0.33
24     0.56  0.27  0.26  0.21 -0.50  0.16

Fraction of years where Δ_cloudy_clear > 0 (per-year terciles):
  phase 10: 2/5  ██

/tmp/atikul-tmpdir-1QD7ds/ipykernel_952441/4207194529.py:58: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  pivot_y = (sub.groupby(['stream','phase','stratum_pyt'])['ae'].mean()
/tmp/atikul-tmpdir-1QD7ds/ipykernel_952441/4207194529.py:59: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  .reset_index().pivot_table(index=['phase','stratum_pyt'],
/tmp/atikul-tmpdir-1QD7ds/ipykernel_952441/4207194529.py:58: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  piv

Median cut: 0.170
Bin sizes: stratum_bin
high    13531
low     13284

S1 marginal value (binary, low=cloudy half):
stratum_bin  high   low  Δ (low − high)
phase                                  
10          -0.14 -0.44           -0.30
12          -0.29  0.09            0.39
15          -0.34  0.56            0.90
18           0.24  0.58            0.34
19           0.23  0.50            0.27
21           0.27  0.30            0.03
24           0.10  0.51            0.42

Per-year robustness (binary):
year   2017  2018  2019  2020  2021  mean
phase                                    
10    -0.16 -0.47 -0.00 -0.52  0.20 -0.19
12    -1.38 -0.18  0.16 -0.13  0.39 -0.23
15     2.08  1.05 -0.10  0.26 -0.10  0.64
18     0.59  0.54  0.17  0.11  0.15  0.31
19     0.68 -0.27 -0.11  0.81 -0.20  0.18
21     0.51 -0.03  0.41  0.30  0.02  0.24
24     0.01  0.31 -0.09  0.84 -0.21  0.17

Fraction of years where Δ_cloudy_clear > 0 (binary):
  phase 10: 1/5  ██████
  phase 12: 2/5  ████████████
  phase 

## 8. Save for figure generation

In [12]:
mae_table.to_csv(OUTPUT_DIR / 'mae_by_stream_phase_stratum.csv')
s1_marginal_df.to_csv(OUTPUT_DIR / 's1_marginal_by_stratum.csv')
all_marginal_df.to_csv(OUTPUT_DIR / 'all_marginal_by_stratum.csv')
print('Saved key tables to', OUTPUT_DIR)


Saved key tables to /fs1/home/atikul/Phenology/phase5_outputs


## 9. Interpretation guide

**What we hope to see:**

For at least the late-season phases (18, 19, 21, 24):
- `s1_marginal_by_stratum` should be POSITIVE (S1 helps overall)
- The "Δ (cloudy − clear)" column should be POSITIVE (S1 helps MORE under clouds)

**What might happen instead:**

- S1's marginal value is uniformly small (e.g., < 0.5 days everywhere) → S1 doesn't
  carry enough phenological signal for fine-grained gains. The honest framing
  becomes "S1 contributes modestly across all conditions, with no detectable
  cloud-cover dependence."

- S1's marginal value is uniformly negative → S1 is actually noise. We'd revisit
  the orbit-mixing limitation and acknowledge it.

- Mixed pattern (helps for some phases, hurts for others, no cloudiness trend) →
  Most likely real-world outcome. Reframes as "S1 contributes differently across
  phases, but not as a function of cloud cover."

Either way, this is a publishable section. The honest result IS the contribution.
